# 07d — CQR Calibration Experiments

Prototype and validate Conformalized Quantile Regression (CQR) on top of the current 07c winner.

**Goal:** Determine if the 07c model's quantile predictions can be effectively calibrated to provide statistically valid prediction intervals.

**Baseline anchor:**
- Dataset: `2026-06-03_008`
- Split: `price_stratified` outer split, group-aware calibration split inside outer train
- Target: `price_egp_log`
- Winner: current 07c winner from `v2_experiment_shortlist.csv`

**Workflow:**
1. Reconstruct `train_fit / val / calibration / test`
2. Train the 07c winner once on `train_fit`
3. Predict raw quantiles on calibration and test
4. Enforce monotonicity
5. Compute nonconformity scores and `q_hat`
6. Apply calibrated intervals on test
7. Validate empirical coverage and export artifacts

## 1) Imports & Configuration

In [1]:
from pathlib import Path
import json
import math
import sys

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'v2_experiment_utils.py').exists():
    NOTEBOOK_DIR = Path('/home/mo-seif/Documents/GP/ml-service/notebooks')
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from v2_experiment_utils import (
    RANDOM_STATE,
    DEFAULT_QUANTILES_5,
    XGB_BASE_PARAMS,
    LGBM_BASE_PARAMS,
    load_processed_dataset,
    build_primary_splits,
    split_train_val_indices,
    build_feature_spec,
    make_lgbm_frame,
    fit_label_encoders,
    transform_with_label_encoders,
    train_lgbm_quantile_models,
    train_xgb_quantile_models,
    predict_lgbm_quantiles,
    predict_xgb_quantiles,
    blend_prediction_maps,
    compute_metrics_egp,
    make_experiment_dir,
    sample_weights_from_price,
)

pd.set_option('display.float_format', lambda value: f'{value:,.4f}')
np.set_printoptions(suppress=True)

In [2]:
DATASET_TAG = '2026-06-03_008'
SPLIT_NAME = 'price_stratified'
TARGET_COL = 'price_egp_log'
FEATURE_LIST = []

CALIBRATION_FRACTION = 0.15
VAL_FRACTION = 0.10
COVERAGE_TARGETS = {'80': 0.80, '90': 0.90}
QUANTILE_KEYS = ['q05', 'q10', 'q50', 'q90', 'q95']

MAX_BOOST_ROUNDS = 1200
EARLY_STOPPING_ROUNDS = 40
FORCE_RECOMPUTE_CQR = False

OUT_DIR = make_experiment_dir('07d_cqr_calibration_experiments')
CACHE_DIR = OUT_DIR / 'cqr_cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

print('Dataset tag:', DATASET_TAG)
print('Split:', SPLIT_NAME)
print('Target:', TARGET_COL)
print('Calibration fraction:', CALIBRATION_FRACTION)
print('Validation fraction inside fit pool:', VAL_FRACTION)
print('Output dir:', OUT_DIR)

Dataset tag: 2026-06-03_008
Split: price_stratified
Target: price_egp_log
Calibration fraction: 0.15
Validation fraction inside fit pool: 0.1
Output dir: /home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07d_cqr_calibration_experiments


In [3]:
def to_price_scale(values: np.ndarray, y_reference: np.ndarray, is_log_target: bool) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    if not is_log_target:
        return arr
    ref = np.asarray(y_reference, dtype=float)
    if float(np.nanmedian(ref)) < 10:
        return np.power(10.0, arr)
    return np.exp(arr)


def pred_map_to_price_scale(pred_map: dict[str, np.ndarray], y_reference: np.ndarray, is_log_target: bool) -> dict[str, np.ndarray]:
    return {
        key: to_price_scale(values, y_reference, is_log_target)
        for key, values in pred_map.items()
    }


def enforce_monotonic_pred_map(pred_map: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    matrix = np.column_stack([np.asarray(pred_map[key], dtype=float) for key in QUANTILE_KEYS])
    matrix = np.sort(matrix, axis=1)
    return {key: matrix[:, idx] for idx, key in enumerate(QUANTILE_KEYS)}


def quantile_frame(pred_map: dict[str, np.ndarray]) -> pd.DataFrame:
    return pd.DataFrame({key: np.asarray(pred_map[key], dtype=float) for key in QUANTILE_KEYS})


def safe_quantile(scores: np.ndarray, level: float) -> float:
    level = min(max(float(level), 0.0), 1.0)
    try:
        return float(np.quantile(scores, level, method='higher'))
    except TypeError:
        return float(np.quantile(scores, level, interpolation='higher'))


def conformal_level(n: int, coverage_target: float) -> float:
    if n <= 0:
        raise ValueError('Calibration size must be positive')
    return min(math.ceil((n + 1) * coverage_target) / n, 1.0)


def price_tier_from_price(price: pd.Series) -> pd.Series:
    return pd.cut(
        pd.to_numeric(price, errors='coerce'),
        bins=[0, 500_000, 1_500_000, 4_000_000, float('inf')],
        labels=['economy', 'standard', 'luxury', 'ultra_luxury'],
        include_lowest=True,
    ).astype(str)


def compute_interval_summary(y_true: np.ndarray, pred_map: dict[str, np.ndarray], is_log_target: bool) -> dict:
    y_eval = to_price_scale(y_true, y_true, is_log_target)
    pred_eval = pred_map_to_price_scale(pred_map, y_true, is_log_target)
    q05 = np.asarray(pred_eval['q05'], dtype=float)
    q10 = np.asarray(pred_eval['q10'], dtype=float)
    q50 = np.asarray(pred_eval['q50'], dtype=float)
    q90 = np.asarray(pred_eval['q90'], dtype=float)
    q95 = np.asarray(pred_eval['q95'], dtype=float)
    width80 = (q90 - q10) / np.maximum(np.abs(q50), 1e-9)
    width90 = (q95 - q05) / np.maximum(np.abs(q50), 1e-9)
    point_metrics = compute_metrics_egp(y_true, pred_map['q50'], is_log=is_log_target)
    return {
        **point_metrics,
        'Coverage_80_pct': float(np.mean((y_eval >= q10) & (y_eval <= q90)) * 100),
        'Coverage_90_pct': float(np.mean((y_eval >= q05) & (y_eval <= q95)) * 100),
        'Width_80_pct': float(np.mean(width80) * 100),
        'Width_90_pct': float(np.mean(width90) * 100),
        'Median_price_egp': float(np.median(y_eval)),
    }


def build_prediction_table(base_df: pd.DataFrame, pred_map_raw: dict[str, np.ndarray], pred_map_monotonic: dict[str, np.ndarray], is_log_target: bool) -> pd.DataFrame:
    raw_price = pred_map_to_price_scale(pred_map_raw, base_df[TARGET_COL].to_numpy(), is_log_target)
    mono_price = pred_map_to_price_scale(pred_map_monotonic, base_df[TARGET_COL].to_numpy(), is_log_target)
    out = base_df[['make', 'model', 'price_egp', TARGET_COL]].copy().reset_index().rename(columns={'index': 'row_index'})
    out['price_tier'] = price_tier_from_price(out['price_egp'])
    for key in QUANTILE_KEYS:
        out[f'raw_{key}'] = raw_price[key]
        out[f'monotonic_{key}'] = mono_price[key]
    return out

In [4]:
def to_price_scale(values: np.ndarray, y_reference: np.ndarray, is_log_target: bool) -> np.ndarray:
    arr = np.asarray(values, dtype=float)
    if not is_log_target:
        return arr
    ref = np.asarray(y_reference, dtype=float)
    if float(np.nanmedian(ref)) < 10:
        return np.power(10.0, arr)
    return np.exp(arr)


def pred_map_to_price_scale(pred_map: dict[str, np.ndarray], y_reference: np.ndarray, is_log_target: bool) -> dict[str, np.ndarray]:
    return {
        key: to_price_scale(values, y_reference, is_log_target)
        for key, values in pred_map.items()
    }

In [5]:
def enforce_monotonic_pred_map(pred_map: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    matrix = np.column_stack([np.asarray(pred_map[key], dtype=float) for key in QUANTILE_KEYS])
    matrix = np.sort(matrix, axis=1)
    return {key: matrix[:, idx] for idx, key in enumerate(QUANTILE_KEYS)}


def quantile_frame(pred_map: dict[str, np.ndarray]) -> pd.DataFrame:
    return pd.DataFrame({key: np.asarray(pred_map[key], dtype=float) for key in QUANTILE_KEYS})

In [6]:
def safe_quantile(scores: np.ndarray, level: float) -> float:
    level = min(max(float(level), 0.0), 1.0)
    try:
        return float(np.quantile(scores, level, method='higher'))
    except TypeError:
        return float(np.quantile(scores, level, interpolation='higher'))


def conformal_level(n: int, coverage_target: float) -> float:
    if n <= 0:
        raise ValueError('Calibration size must be positive')
    return min(math.ceil((n + 1) * coverage_target) / n, 1.0)

In [7]:
def price_tier_from_price(price: pd.Series) -> pd.Series:
    return pd.cut(
        pd.to_numeric(price, errors='coerce'),
        bins=[0, 500_000, 1_500_000, 4_000_000, float('inf')],
        labels=['economy', 'standard', 'luxury', 'ultra_luxury'],
        include_lowest=True,
    ).astype(str)

In [8]:
def compute_interval_summary(y_true: np.ndarray, pred_map: dict[str, np.ndarray], is_log_target: bool) -> dict:
    y_eval = to_price_scale(y_true, y_true, is_log_target)
    pred_eval = pred_map_to_price_scale(pred_map, y_true, is_log_target)
    q05 = np.asarray(pred_eval['q05'], dtype=float)
    q10 = np.asarray(pred_eval['q10'], dtype=float)
    q50 = np.asarray(pred_eval['q50'], dtype=float)
    q90 = np.asarray(pred_eval['q90'], dtype=float)
    q95 = np.asarray(pred_eval['q95'], dtype=float)
    width80 = (q90 - q10) / np.maximum(np.abs(q50), 1e-9)
    width90 = (q95 - q05) / np.maximum(np.abs(q50), 1e-9)
    point_metrics = compute_metrics_egp(y_true, pred_map['q50'], is_log=is_log_target)
    return {
        **point_metrics,
        'Coverage_80_pct': float(np.mean((y_eval >= q10) & (y_eval <= q90)) * 100),
        'Coverage_90_pct': float(np.mean((y_eval >= q05) & (y_eval <= q95)) * 100),
        'Width_80_pct': float(np.mean(width80) * 100),
        'Width_90_pct': float(np.mean(width90) * 100),
        'Median_price_egp': float(np.median(y_eval)),
    }


def build_prediction_table(base_df: pd.DataFrame, pred_map_raw: dict[str, np.ndarray], pred_map_monotonic: dict[str, np.ndarray], is_log_target: bool) -> pd.DataFrame:
    raw_price = pred_map_to_price_scale(pred_map_raw, base_df[TARGET_COL].to_numpy(), is_log_target)
    mono_price = pred_map_to_price_scale(pred_map_monotonic, base_df[TARGET_COL].to_numpy(), is_log_target)
    out = base_df[['make', 'model', 'price_egp', TARGET_COL]].copy().reset_index().rename(columns={'index': 'row_index'})
    out['price_tier'] = price_tier_from_price(out['price_egp'])
    for key in QUANTILE_KEYS:
        out[f'raw_{key}'] = raw_price[key]
        out[f'monotonic_{key}'] = mono_price[key]
    return out

## 2) Load Winner Metadata

In [9]:
SHORTLIST_PATH = Path('/home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07c_model_v2_training_experiments/v2_experiment_shortlist.csv')
if not SHORTLIST_PATH.exists():
    raise FileNotFoundError(f'07c shortlist not found at: {SHORTLIST_PATH}')

shortlist_df = pd.read_csv(SHORTLIST_PATH)
best_v2 = shortlist_df.sort_values(['MAPE_pct', 'MAE']).iloc[0].to_dict()
best_exp_name = str(best_v2['experiment_name'])
best_framework = str(best_v2['framework'])
best_use_tuned = bool(best_v2['use_tuned'])
best_use_weights = bool(best_v2['use_weights'])

if best_use_tuned:
    raise RuntimeError(
        'The current 07c winner requires tuned params that are not exported as reusable artifacts. '
        'This notebook currently supports the present untuned winner path only.'
    )

winner_summary_df = pd.DataFrame([best_v2])
print(f'Winner: {best_exp_name}')
print(f'Framework: {best_framework}')
print(f'Use tuned params: {best_use_tuned}')
print(f'Use sample weights: {best_use_weights}')
display(winner_summary_df)

Winner: ensemble_baseline_5q
Framework: WeightedEnsemble
Use tuned params: False
Use sample weights: False


,experiment_name,use_weights,use_tuned,quantile_count,framework,MAE,RMSE,R2,MAPE_pct,Within_10pct,Within_15pct,Coverage_80_pct,Coverage_90_pct,Mean_width_pct
0,ensemble_baseline_5q,False,False,5,WeightedEnsemble,"151,004.7090","602,399.1851",0.8887,11.2937,61.6400,76.8800,73.5200,86.5200,36.2937


## 3) Reconstruct CQR Data Splits

In [10]:
df_full = load_processed_dataset(DATASET_TAG)
is_log_target = TARGET_COL == 'price_egp_log'
NUM_COLS, CAT_COLS, FEATURE_COLS = build_feature_spec(FEATURE_LIST)

split_map = build_primary_splits(df_full)
outer_train_idx, test_idx = split_map[SPLIT_NAME]
outer_train_df = df_full.loc[outer_train_idx].copy()
test_df = df_full.loc[test_idx].copy()

groups_outer = outer_train_df['make'].fillna('UNKNOWN').astype(str) + '__' + outer_train_df['model'].fillna('UNKNOWN').astype(str)
calibration_splitter = GroupShuffleSplit(n_splits=1, test_size=CALIBRATION_FRACTION, random_state=RANDOM_STATE)
fit_pool_pos, calibration_pos = next(calibration_splitter.split(outer_train_df, groups=groups_outer))
fit_pool_idx = outer_train_df.iloc[fit_pool_pos].index.to_numpy()
calibration_idx = outer_train_df.iloc[calibration_pos].index.to_numpy()

train_fit_idx, val_idx = split_train_val_indices(fit_pool_idx, val_fraction=VAL_FRACTION, random_state=RANDOM_STATE)

train_fit_df = df_full.loc[train_fit_idx].copy()
val_df = df_full.loc[val_idx].copy()
calibration_df = df_full.loc[calibration_idx].copy()

def split_summary_row(name: str, frame: pd.DataFrame) -> dict:
    groups = frame['make'].fillna('UNKNOWN').astype(str) + '__' + frame['model'].fillna('UNKNOWN').astype(str)
    return {
        'split': name,
        'rows': len(frame),
        'unique_make_model_groups': int(groups.nunique()),
        'median_price_egp': float(frame['price_egp'].median()),
        'min_price_egp': float(frame['price_egp'].min()),
        'max_price_egp': float(frame['price_egp'].max()),
    }

split_summary_df = pd.DataFrame([
    split_summary_row('train_fit', train_fit_df),
    split_summary_row('val', val_df),
    split_summary_row('calibration', calibration_df),
    split_summary_row('test', test_df),
])

display(split_summary_df)

,split,rows,unique_make_model_groups,median_price_egp,min_price_egp,max_price_egp
0,train_fit,7875,374,"720,000.0000","28,000.0000","19,500,000.0000"
1,val,875,258,"730,000.0000","25,000.0000","18,800,000.0000"
2,calibration,1248,67,"722,500.0000","70,000.0000","20,000,000.0000"
3,test,2500,393,"725,000.0000","35,000.0000","20,000,000.0000"


In [11]:
def group_set(frame: pd.DataFrame) -> set[str]:
    return set(frame['make'].fillna('UNKNOWN').astype(str) + '__' + frame['model'].fillna('UNKNOWN').astype(str))

train_groups = group_set(train_fit_df)
val_groups = group_set(val_df)
cal_groups = group_set(calibration_df)
test_groups = group_set(test_df)

overlap_df = pd.DataFrame([
    {'pair': 'train_fit ∩ calibration', 'shared_groups': len(train_groups & cal_groups)},
    {'pair': 'train_fit ∩ test', 'shared_groups': len(train_groups & test_groups)},
    {'pair': 'calibration ∩ test', 'shared_groups': len(cal_groups & test_groups)},
    {'pair': 'val ∩ calibration', 'shared_groups': len(val_groups & cal_groups)},
])

tier_distribution_df = pd.concat([
    train_fit_df.assign(split='train_fit'),
    val_df.assign(split='val'),
    calibration_df.assign(split='calibration'),
    test_df.assign(split='test'),
], axis=0)
tier_distribution_df['price_tier'] = price_tier_from_price(tier_distribution_df['price_egp'])
tier_distribution_df = (
    tier_distribution_df.groupby(['split', 'price_tier']).size().rename('rows').reset_index()
    .sort_values(['split', 'price_tier']).reset_index(drop=True)
)

display(overlap_df)
display(tier_distribution_df)

,pair,shared_groups
0,train_fit ∩ calibration,0
1,train_fit ∩ test,332
2,calibration ∩ test,61
3,val ∩ calibration,0


,split,price_tier,rows
0,calibration,economy,463
1,calibration,luxury,165
2,calibration,standard,520
3,calibration,ultra_luxury,100
4,test,economy,906
5,test,luxury,461
6,test,standard,966
7,test,ultra_luxury,167
8,train_fit,economy,2853
9,train_fit,luxury,1473


## 4) Prepare Feature Matrices and Train the 07c Winner

In [12]:
X_train_lgb = make_lgbm_frame(train_fit_df, FEATURE_COLS, CAT_COLS)
X_val_lgb = make_lgbm_frame(val_df, FEATURE_COLS, CAT_COLS)
X_cal_lgb = make_lgbm_frame(calibration_df, FEATURE_COLS, CAT_COLS)
X_test_lgb = make_lgbm_frame(test_df, FEATURE_COLS, CAT_COLS)

label_encoders = fit_label_encoders(train_fit_df, CAT_COLS)
X_train_xgb = transform_with_label_encoders(train_fit_df, FEATURE_COLS, CAT_COLS, label_encoders)
X_val_xgb = transform_with_label_encoders(val_df, FEATURE_COLS, CAT_COLS, label_encoders)
X_cal_xgb = transform_with_label_encoders(calibration_df, FEATURE_COLS, CAT_COLS, label_encoders)
X_test_xgb = transform_with_label_encoders(test_df, FEATURE_COLS, CAT_COLS, label_encoders)

train_weights = sample_weights_from_price(train_fit_df['price_egp']) if best_use_weights else None
val_weights = sample_weights_from_price(val_df['price_egp']) if best_use_weights else None

print('Feature count:', len(FEATURE_COLS))
print('LightGBM frames:', X_train_lgb.shape, X_val_lgb.shape, X_cal_lgb.shape, X_test_lgb.shape)
print('XGBoost frames: ', X_train_xgb.shape, X_val_xgb.shape, X_cal_xgb.shape, X_test_xgb.shape)
print('Using sample weights:', best_use_weights)

Feature count: 15
LightGBM frames: (7875, 15) (875, 15) (1248, 15) (2500, 15)
XGBoost frames:  (7875, 15) (875, 15) (1248, 15) (2500, 15)
Using sample weights: False


In [13]:
raw_calibration_cache_path = CACHE_DIR / 'raw_calibration_predictions.csv'
raw_test_cache_path = CACHE_DIR / 'raw_test_predictions.csv'
raw_metadata_cache_path = CACHE_DIR / 'raw_prediction_metadata.json'


def train_and_predict_current_winner() -> tuple[dict[str, np.ndarray], dict[str, np.ndarray]]:
    quantiles = DEFAULT_QUANTILES_5
    xgb_params = {**XGB_BASE_PARAMS, 'seed': RANDOM_STATE}
    lgbm_params = {**LGBM_BASE_PARAMS, 'seed': RANDOM_STATE}

    if best_framework == 'WeightedEnsemble':
        xgb_models = train_xgb_quantile_models(
            X_train_xgb,
            train_fit_df[TARGET_COL],
            X_val_xgb,
            val_df[TARGET_COL],
            quantiles,
            params=xgb_params,
            n_estimators_max=MAX_BOOST_ROUNDS,
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            sample_weight_train=train_weights,
            sample_weight_valid=val_weights,
        )
        lgbm_models = train_lgbm_quantile_models(
            X_train_lgb,
            train_fit_df[TARGET_COL],
            X_val_lgb,
            val_df[TARGET_COL],
            CAT_COLS,
            quantiles,
            params=lgbm_params,
            n_estimators_max=MAX_BOOST_ROUNDS,
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            sample_weight_train=train_weights,
            sample_weight_valid=val_weights,
        )
        cal_xgb = predict_xgb_quantiles(xgb_models, X_cal_xgb)
        test_xgb = predict_xgb_quantiles(xgb_models, X_test_xgb)
        cal_lgbm = predict_lgbm_quantiles(lgbm_models, X_cal_lgb)
        test_lgbm = predict_lgbm_quantiles(lgbm_models, X_test_lgb)
        return (
            blend_prediction_maps([cal_xgb, cal_lgbm], weights=[0.55, 0.45]),
            blend_prediction_maps([test_xgb, test_lgbm], weights=[0.55, 0.45]),
        )

    if best_framework == 'LightGBM':
        lgbm_models = train_lgbm_quantile_models(
            X_train_lgb,
            train_fit_df[TARGET_COL],
            X_val_lgb,
            val_df[TARGET_COL],
            CAT_COLS,
            quantiles,
            params=lgbm_params,
            n_estimators_max=MAX_BOOST_ROUNDS,
            early_stopping_rounds=EARLY_STOPPING_ROUNDS,
            sample_weight_train=train_weights,
            sample_weight_valid=val_weights,
        )
        return predict_lgbm_quantiles(lgbm_models, X_cal_lgb), predict_lgbm_quantiles(lgbm_models, X_test_lgb)

    xgb_models = train_xgb_quantile_models(
        X_train_xgb,
        train_fit_df[TARGET_COL],
        X_val_xgb,
        val_df[TARGET_COL],
        quantiles,
        params=xgb_params,
        n_estimators_max=MAX_BOOST_ROUNDS,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        sample_weight_train=train_weights,
        sample_weight_valid=val_weights,
    )
    return predict_xgb_quantiles(xgb_models, X_cal_xgb), predict_xgb_quantiles(xgb_models, X_test_xgb)


if raw_calibration_cache_path.exists() and raw_test_cache_path.exists() and not FORCE_RECOMPUTE_CQR:
    raw_cal_df = pd.read_csv(raw_calibration_cache_path)
    raw_test_df = pd.read_csv(raw_test_cache_path)
else:
    raw_calibration_pred_map, raw_test_pred_map = train_and_predict_current_winner()
    raw_cal_df = quantile_frame(raw_calibration_pred_map)
    raw_test_df = quantile_frame(raw_test_pred_map)
    raw_cal_df.to_csv(raw_calibration_cache_path, index=False)
    raw_test_df.to_csv(raw_test_cache_path, index=False)
    with raw_metadata_cache_path.open('w', encoding='utf-8') as handle:
        json.dump(
            {
                'winner_experiment_name': best_exp_name,
                'framework': best_framework,
                'use_weights': best_use_weights,
                'use_tuned': best_use_tuned,
                'train_fit_rows': int(len(train_fit_df)),
                'val_rows': int(len(val_df)),
                'calibration_rows': int(len(calibration_df)),
                'test_rows': int(len(test_df)),
            },
            handle,
            indent=2,
        )

raw_calibration_pred_map = {key: raw_cal_df[key].to_numpy(dtype=float) for key in QUANTILE_KEYS}
raw_test_pred_map = {key: raw_test_df[key].to_numpy(dtype=float) for key in QUANTILE_KEYS}

print('Raw prediction caches ready:')
print('  calibration:', raw_calibration_cache_path)
print('  test:', raw_test_cache_path)

Raw prediction caches ready:
  calibration: /home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07d_cqr_calibration_experiments/cqr_cache/raw_calibration_predictions.csv
  test: /home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07d_cqr_calibration_experiments/cqr_cache/raw_test_predictions.csv


## 5) Enforce Monotonic Quantiles and Inspect Raw Diagnostics

In [14]:
monotonic_calibration_pred_map = enforce_monotonic_pred_map(raw_calibration_pred_map)
monotonic_test_pred_map = enforce_monotonic_pred_map(raw_test_pred_map)

calibration_prediction_table = build_prediction_table(
    calibration_df,
    raw_calibration_pred_map,
    monotonic_calibration_pred_map,
    is_log_target,
)
test_prediction_table = build_prediction_table(
    test_df,
    raw_test_pred_map,
    monotonic_test_pred_map,
    is_log_target,
)

raw_summary_df = pd.DataFrame([
    {
        'split': 'calibration',
        'mode': 'raw_monotonic',
        **compute_interval_summary(calibration_df[TARGET_COL].to_numpy(), monotonic_calibration_pred_map, is_log_target),
    },
    {
        'split': 'test',
        'mode': 'raw_monotonic',
        **compute_interval_summary(test_df[TARGET_COL].to_numpy(), monotonic_test_pred_map, is_log_target),
    },
])

display(raw_summary_df)
display(calibration_prediction_table.head())
display(test_prediction_table.head())

,split,mode,MAE,RMSE,R2,MAPE_pct,Within_10pct,Within_15pct,Coverage_80_pct,Coverage_90_pct,Width_80_pct,Width_90_pct,Median_price_egp
0,calibration,raw_monotonic,"318,033.4555","908,720.2792",0.8166,21.9614,38.2212,54.4071,60.0160,74.5192,42.9741,65.1843,"722,500.0000"
1,test,raw_monotonic,"168,598.2610","663,751.7517",0.8649,12.2548,59.5200,75.1600,71.6400,85.9200,36.3441,61.3206,"725,000.0000"


,row_index,make,model,price_egp,price_egp_log,price_tier,raw_q05,monotonic_q05,raw_q10,monotonic_q10,raw_q50,monotonic_q50,raw_q90,monotonic_q90,raw_q95,monotonic_q95
0,5336,Fiat,Tipo,800000,13.5924,standard,"626,224.2841","626,224.2841","665,599.1121","665,599.1121","752,534.4900","752,534.4900","926,480.3683","926,480.3683","993,998.6051","993,998.6051"
1,2376,Speranza,A516,220000,12.3014,economy,"288,283.8738","288,283.8738","301,150.1879","301,150.1879","347,112.1835","347,112.1835","439,767.4646","439,767.4646","495,268.3302","495,268.3302"
2,4633,Mercedes,GLE450,6550000,15.6950,ultra_luxury,"1,819,050.4139","1,819,050.4139","2,570,952.0571","2,570,952.0571","4,578,404.6706","4,578,404.6706","5,812,377.5787","5,812,377.5787","6,459,135.2695","6,459,135.2695"
3,1240,Geely,Emgrand 7,410000,12.9239,economy,"549,070.1111","549,070.1111","574,869.7295","574,869.7295","648,810.9524","648,810.9524","811,060.2464","811,060.2464","939,355.6588","939,355.6588"
4,3577,Speranza,A516,230000,12.3458,economy,"203,363.1317","203,363.1317","228,410.7081","228,410.7081","262,833.6072","262,833.6072","336,802.7123","336,802.7123","365,867.1280","365,867.1280"


,row_index,make,model,price_egp,price_egp_log,price_tier,raw_q05,monotonic_q05,raw_q10,monotonic_q10,raw_q50,monotonic_q50,raw_q90,monotonic_q90,raw_q95,monotonic_q95
0,7321,Kia,Cerato,590000,13.2879,standard,"457,385.4693","457,385.4693","512,840.6487","512,840.6487","589,226.2640","589,226.2640","630,664.9337","630,664.9337","674,965.5436","674,965.5436"
1,5927,Audi,Q3,2400000,14.6910,luxury,"1,475,447.6943","1,475,447.6943","1,916,819.6131","1,916,819.6131","2,421,687.3076","2,421,687.3076","2,793,855.4599","2,702,683.1981","2,702,683.1981","2,793,855.4599"
2,8151,Renault,Megane,450000,13.0170,economy,"462,500.2201","462,500.2201","502,777.0538","502,777.0538","612,776.0739","612,776.0739","699,216.3621","699,216.3621","733,637.8150","733,637.8150"
3,8056,Hyundai,Accent RB,750000,13.5278,standard,"683,218.0785","683,218.0785","690,879.8456","690,879.8456","785,474.8327","785,474.8327","968,960.6636","968,960.6636","1,082,377.9352","1,082,377.9352"
4,1231,Chery,Envy,330000,12.7068,economy,"309,659.5273","309,659.5273","310,230.3764","310,230.3764","330,741.7885","330,741.7885","404,960.2162","404,960.2162","493,053.9407","493,053.9407"


## 6) Compute Conformal Scores and Correction Factors

In [15]:
y_cal_price = calibration_df['price_egp'].to_numpy(dtype=float)
y_test_price = test_df['price_egp'].to_numpy(dtype=float)
calibration_pred_price = pred_map_to_price_scale(monotonic_calibration_pred_map, calibration_df[TARGET_COL].to_numpy(), is_log_target)
test_pred_price = pred_map_to_price_scale(monotonic_test_pred_map, test_df[TARGET_COL].to_numpy(), is_log_target)

scores_80 = np.maximum(calibration_pred_price['q10'] - y_cal_price, y_cal_price - calibration_pred_price['q90'])
scores_90 = np.maximum(calibration_pred_price['q05'] - y_cal_price, y_cal_price - calibration_pred_price['q95'])

n_cal = len(calibration_df)
level_80 = conformal_level(n_cal, COVERAGE_TARGETS['80'])
level_90 = conformal_level(n_cal, COVERAGE_TARGETS['90'])
q_hat_80 = safe_quantile(scores_80, level_80)
q_hat_90 = safe_quantile(scores_90, level_90)

score_summary_df = pd.DataFrame([
    {
        'interval': '80',
        'calibration_n': n_cal,
        'coverage_target': COVERAGE_TARGETS['80'],
        'quantile_level_used': level_80,
        'q_hat': q_hat_80,
        'score_min': float(np.min(scores_80)),
        'score_median': float(np.median(scores_80)),
        'score_p90': float(np.quantile(scores_80, 0.90)),
        'score_p95': float(np.quantile(scores_80, 0.95)),
        'score_max': float(np.max(scores_80)),
    },
    {
        'interval': '90',
        'calibration_n': n_cal,
        'coverage_target': COVERAGE_TARGETS['90'],
        'quantile_level_used': level_90,
        'q_hat': q_hat_90,
        'score_min': float(np.min(scores_90)),
        'score_median': float(np.median(scores_90)),
        'score_p90': float(np.quantile(scores_90, 0.90)),
        'score_p95': float(np.quantile(scores_90, 0.95)),
        'score_max': float(np.max(scores_90)),
    },
])

display(score_summary_df)

,interval,calibration_n,coverage_target,quantile_level_used,q_hat,score_min,score_median,score_p90,score_p95,score_max
0,80,1248,0.8000,0.8013,"67,114.7016","-2,156,054.6093","-21,943.8391","184,760.9327","734,027.6679","8,348,009.2775"
1,90,1248,0.9000,0.9014,"85,323.7730","-3,229,423.1046","-60,675.6008","83,695.2410","255,710.6151","6,743,830.5743"


## 7) Apply CQR on Test and Compare Raw vs Calibrated

In [16]:
calibrated_test_pred_price = {key: np.asarray(values, dtype=float).copy() for key, values in test_pred_price.items()}
calibrated_test_pred_price['q10'] = np.clip(test_pred_price['q10'] - q_hat_80, a_min=0.0, a_max=None)
calibrated_test_pred_price['q90'] = test_pred_price['q90'] + q_hat_80
calibrated_test_pred_price['q05'] = np.clip(test_pred_price['q05'] - q_hat_90, a_min=0.0, a_max=None)
calibrated_test_pred_price['q95'] = test_pred_price['q95'] + q_hat_90
calibrated_test_pred_price = enforce_monotonic_pred_map(calibrated_test_pred_price)

raw_test_summary = compute_interval_summary(test_df[TARGET_COL].to_numpy(), monotonic_test_pred_map, is_log_target)
calibrated_test_summary = compute_interval_summary(test_df['price_egp'].to_numpy(), calibrated_test_pred_price, False)
comparison_summary_df = pd.DataFrame([
    {'split': 'test', 'mode': 'raw_monotonic', **raw_test_summary},
    {'split': 'test', 'mode': 'cqr_calibrated', **calibrated_test_summary},
])

test_export_df = test_prediction_table.copy()
test_export_df['calibrated_lower_80'] = calibrated_test_pred_price['q10']
test_export_df['calibrated_upper_80'] = calibrated_test_pred_price['q90']
test_export_df['calibrated_lower_90'] = calibrated_test_pred_price['q05']
test_export_df['calibrated_upper_90'] = calibrated_test_pred_price['q95']

display(comparison_summary_df)
display(test_export_df.head())

,split,mode,MAE,RMSE,R2,MAPE_pct,Within_10pct,Within_15pct,Coverage_80_pct,Coverage_90_pct,Width_80_pct,Width_90_pct,Median_price_egp
0,test,raw_monotonic,"168,598.2610","663,751.7517",0.8649,12.2548,59.5200,75.1600,71.6400,85.9200,36.3441,61.3206,"725,000.0000"
1,test,cqr_calibrated,"168,598.2610","663,751.7517",0.8649,12.2548,59.5200,75.1600,91.0800,96.3200,64.1941,96.3881,"725,000.0000"


,row_index,make,model,price_egp,price_egp_log,price_tier,raw_q05,monotonic_q05,raw_q10,monotonic_q10,raw_q50,monotonic_q50,raw_q90,monotonic_q90,raw_q95,monotonic_q95,calibrated_lower_80,calibrated_upper_80,calibrated_lower_90,calibrated_upper_90
0,7321,Kia,Cerato,590000,13.2879,standard,"457,385.4693","457,385.4693","512,840.6487","512,840.6487","589,226.2640","589,226.2640","630,664.9337","630,664.9337","674,965.5436","674,965.5436","445,725.9471","697,779.6353","372,061.6963","760,289.3166"
1,5927,Audi,Q3,2400000,14.6910,luxury,"1,475,447.6943","1,475,447.6943","1,916,819.6131","1,916,819.6131","2,421,687.3076","2,421,687.3076","2,793,855.4599","2,702,683.1981","2,702,683.1981","2,793,855.4599","1,849,704.9115","2,769,797.8997","1,390,123.9213","2,879,179.2329"
2,8151,Renault,Megane,450000,13.0170,economy,"462,500.2201","462,500.2201","502,777.0538","502,777.0538","612,776.0739","612,776.0739","699,216.3621","699,216.3621","733,637.8150","733,637.8150","435,662.3522","766,331.0638","377,176.4471","818,961.5880"
3,8056,Hyundai,Accent RB,750000,13.5278,standard,"683,218.0785","683,218.0785","690,879.8456","690,879.8456","785,474.8327","785,474.8327","968,960.6636","968,960.6636","1,082,377.9352","1,082,377.9352","623,765.1440","1,036,075.3652","597,894.3055","1,167,701.7082"
4,1231,Chery,Envy,330000,12.7068,economy,"309,659.5273","309,659.5273","310,230.3764","310,230.3764","330,741.7885","330,741.7885","404,960.2162","404,960.2162","493,053.9407","493,053.9407","243,115.6748","472,074.9179","224,335.7542","578,377.7137"


## 8) Optional Per-Tier Diagnostic View

In [17]:
tier_rows = []
for tier_name, tier_group in test_export_df.groupby('price_tier', dropna=False):
    idx = tier_group.index.to_numpy()
    if len(idx) == 0:
        continue
    raw_tier_pred_map = {key: test_pred_price[key][idx] for key in QUANTILE_KEYS}
    calibrated_tier_pred_map = {
        'q05': calibrated_test_pred_price['q05'][idx],
        'q10': calibrated_test_pred_price['q10'][idx],
        'q50': calibrated_test_pred_price['q50'][idx],
        'q90': calibrated_test_pred_price['q90'][idx],
        'q95': calibrated_test_pred_price['q95'][idx],
    }
    raw_tier_summary = compute_interval_summary(tier_group['price_egp'].to_numpy(), raw_tier_pred_map, False)
    calibrated_tier_summary = compute_interval_summary(tier_group['price_egp'].to_numpy(), calibrated_tier_pred_map, False)
    tier_rows.append({'price_tier': tier_name, 'mode': 'raw_monotonic', 'rows': len(tier_group), **raw_tier_summary})
    tier_rows.append({'price_tier': tier_name, 'mode': 'cqr_calibrated', 'rows': len(tier_group), **calibrated_tier_summary})

per_tier_summary_df = pd.DataFrame(tier_rows).sort_values(['price_tier', 'mode']).reset_index(drop=True)
display(per_tier_summary_df)

,price_tier,mode,rows,MAE,RMSE,R2,MAPE_pct,Within_10pct,Within_15pct,Coverage_80_pct,Coverage_90_pct,Width_80_pct,Width_90_pct,Median_price_egp
0,economy,cqr_calibrated,906,"40,992.3712","78,029.9007",0.5658,16.4332,48.5651,63.9073,97.4614,99.4481,100.0971,147.9262,"285,000.0000"
1,economy,raw_monotonic,906,"40,992.3712","78,029.9007",0.5658,16.4332,48.5651,63.9073,70.0883,84.6578,45.0677,78.9006,"285,000.0000"
2,luxury,cqr_calibrated,461,"225,713.0633","370,102.5079",0.7339,9.5990,67.4620,81.3449,85.6833,93.2755,36.4976,60.8957,"2,250,000.0000"
3,luxury,raw_monotonic,461,"225,713.0633","370,102.5079",0.7339,9.5990,67.4620,81.3449,74.1866,87.4187,30.0071,52.6443,"2,250,000.0000"
4,standard,cqr_calibrated,966,"81,553.5529","137,211.7214",0.7456,9.1772,66.6667,83.7474,91.2008,97.3085,45.6527,67.5800,"830,000.0000"
5,standard,raw_monotonic,966,"81,553.5529","137,211.7214",0.7456,9.1772,66.6667,83.7474,72.7743,87.7847,28.7422,46.0815,"830,000.0000"
6,ultra_luxury,cqr_calibrated,167,"1,206,719.2205","2,464,802.3178",0.2732,14.7201,55.6886,69.4611,70.6587,82.0359,53.1212,81.4001,"5,800,000.0000"
7,ultra_luxury,raw_monotonic,167,"1,206,719.2205","2,464,802.3178",0.2732,14.7201,55.6886,69.4611,66.4671,77.8443,50.4832,78.0463,"5,800,000.0000"


## 9) Export Artifacts

In [ ]:
summary_export_path = OUT_DIR / 'cqr_summary.csv'
config_export_path = OUT_DIR / 'cqr_calibration_config.json'
calibration_export_path = OUT_DIR / 'cqr_calibration_predictions.csv'
test_export_path = OUT_DIR / 'cqr_test_predictions.csv'
per_tier_export_path = OUT_DIR / 'cqr_per_tier_summary.csv'

summary_export_df = pd.concat([
    raw_summary_df.loc[raw_summary_df['split'] == 'calibration'],
    comparison_summary_df,
], axis=0, ignore_index=True)
summary_export_df.to_csv(summary_export_path, index=False)
calibration_prediction_table.to_csv(calibration_export_path, index=False)
test_export_df.to_csv(test_export_path, index=False)
if not per_tier_summary_df.empty:
    per_tier_summary_df.to_csv(per_tier_export_path, index=False)

cqr_config = {
    'winner_experiment_name': best_exp_name,
    'framework': best_framework,
    'use_weights': best_use_weights,
    'use_tuned': best_use_tuned,
    'dataset_tag': DATASET_TAG,
    'split_name': SPLIT_NAME,
    'target_col': TARGET_COL,
    'calibration_fraction': CALIBRATION_FRACTION,
    'val_fraction': VAL_FRACTION,
    'split_sizes': {
        'train_fit': int(len(train_fit_df)),
        'val': int(len(val_df)),
        'calibration': int(len(calibration_df)),
        'test': int(len(test_df)),
    },
    'coverage_targets': COVERAGE_TARGETS,
    'conformal_quantile_levels': {
        '80': level_80,
        '90': level_90,
    },
    'q_hat_80': float(q_hat_80),
    'q_hat_90': float(q_hat_90),
    'raw_test_coverage_80_pct': float(raw_test_summary['Coverage_80_pct']),
    'raw_test_coverage_90_pct': float(raw_test_summary['Coverage_90_pct']),
    'calibrated_test_coverage_80_pct': float(calibrated_test_summary['Coverage_80_pct']),
    'calibrated_test_coverage_90_pct': float(calibrated_test_summary['Coverage_90_pct']),
    'raw_test_width_80_pct': float(raw_test_summary['Width_80_pct']),
    'raw_test_width_90_pct': float(raw_test_summary['Width_90_pct']),
    'calibrated_test_width_80_pct': float(calibrated_test_summary['Width_80_pct']),
    'calibrated_test_width_90_pct': float(calibrated_test_summary['Width_90_pct']),
    'method': 'CQR',
    'score_space': 'price_egp',
}
with config_export_path.open('w', encoding='utf-8') as handle:
    json.dump(cqr_config, handle, indent=2)

print('Saved artifacts:')
print(' ', summary_export_path)
print(' ', config_export_path)
print(' ', calibration_export_path)
print(' ', test_export_path)
if per_tier_export_path.exists():
    print(' ', per_tier_export_path)

Saved artifacts:
  /home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07d_cqr_calibration_experiments/cqr_summary.csv
  /home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07d_cqr_calibration_experiments/cqr_calibration_config.json
  /home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07d_cqr_calibration_experiments/cqr_calibration_predictions.csv
  /home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07d_cqr_calibration_experiments/cqr_test_predictions.csv
  /home/mo-seif/Documents/GP/ml-service/models/experiments/plan2_plan4/07d_cqr_calibration_experiments/cqr_per_tier_summary.csv


## 10) Interpretation

Use the comparison table and exported config to decide whether the current 07c winner is a good Plan 6 candidate.

Recommended readout:
- If calibrated test coverage lands near 80% / 90% and width inflation is moderate, the winner is a strong CQR base.
- If `q_hat` is very large, the raw quantiles are too under-covered and Plan 4 training quality likely needs more work before production Plan 6 integration.
- If global coverage looks good but per-tier coverage is uneven, consider per-tier calibration only as a follow-up experiment, not in the first integration pass.